## ModelEU27
Powered by [Eleonora Priori](https://eleonorapriori.github.io/website/) and [Pietro Terna](https://terna.to.it/) 


In [1]:
def making924table(seed, rankNum):
    tab = [[0] * rankNum for _ in range(924)]
    random.seed(seed)
    for i in range(924):
        tab[i][random.randint(0,rankNum-1)] = 1
        
    return tab   

In [2]:
import ctypes
import subprocess
import tempfile
import platform
from pathlib import Path

from mpi4py import MPI
from repast4py import context as ctx
import repast4py 
from repast4py import parameters
from repast4py import schedule
from repast4py import core
from typing import Tuple, List, Dict
import numpy as np
import pandas as pd
import pickle
import random
import csv
import os
import sys
import time
import sectors
from itertools import accumulate

comm = MPI.COMM_WORLD
rank    = comm.Get_rank()
rankNum = comm.Get_size() 

if rankNum > 0: print('rank', rank, 'starting',flush=True)

# create the context to hold the agents and manage cross process
# synchronization
context = ctx.SharedContext(comm)

# Initialize the default schedule runner, HERE to create the t() function,
# returning the tick value
runner = schedule.init_schedule_runner(comm)

#Initializes the repast4py.parameters.params dictionary with the model input parameters.
params = parameters.init_params("modelEU27.yaml", "")

#making the table splitting the creation of firms in ranks
seed924 = params['seed924'] #special independent seed for the 915 table
tab924 = making924table(seed924,rankNum)

#generate random seed
repast4py.random.init(rng_seed=params['myRandom.seed'][rank]) #each rank has a seed
rng = repast4py.random.default_rng # repast seed
rng_np = np.random.default_rng(params['myRandom.seed'][rank]) # numpy seed
random.seed = params['myRandom.seed'][rank]

#timer T()
startTime=-1
def T():
    global startTime
    if startTime < 0:
        startTime=time.time()
    return time.time() - startTime
T() #launches the timer

#cpuTimer Tc()
startCpuTime=-1
def Tc():
    global startCpuTime
    if startCpuTime < 0:
        startCpuTime=time.process_time()
    return time.process_time() - startCpuTime
Tc() #launches the cpu timer

#read control key from firm-features-generation
with open("control4text_ff.txt", "r") as f:
    control = int(f.read())

rank 0 starting


In [3]:
class Firm(core.Agent):

    TYPE = 0
    sharedOutputWarehouse = None
    sharedTargetAddedValue = None
    
    def __init__(self, local_id: int, rank: int, sector: int, labor: int, capital: float, capitalR: float, wage: float,\
                 country: int, recipe: float, countryRecipe: float, laborProductivity: float, avgAssetsUsefulLife: float,\
                 avgPlannedMarkup: float, productionType: str):
                 # orderObservationFrequency: int, 
                 
        super().__init__(id=local_id, type=Firm.TYPE, rank=rank) #uid
    
        self.sector=sector 
        self.labor=labor
        self.capital=capital
        self.capitalR=capitalR
        self.wage=wage
        self.country = country
        #self.investment = investment # serviva per interagire col planner ?
        self.recipe = recipe
        self.countryRecipe = countryRecipe
        self.laborProductivity=laborProductivity
        self.assetsUsefulLife=avgAssetsUsefulLife
        self.plannedMarkup=avgPlannedMarkup
        self.productionType = productionType
        self.address = -1
        self.intermediateWarehouse = [0] * 64
        self.intermediateWarehouseRefill= [0] * 64
        self._targetAddedValue = 0.0
        self.actualAddedValue = 0 
        self.warehouseForPotentialExtraEU27export = 0

        # here we introduce an estimated value of added value in t-1 to generate the intermediate warehouse quantities available at t=0
        self.estimatedAddedValueTminus1 = float(self.labor * self.wage + self.capital * 1000 * self.capitalR) / 12
        self.estimatedAddedValueTminus1 *= self.plannedMarkup
        
        # self.orderObservationFrequency=orderObservationFrequency

    @property
    def outputWarehouse(self):
        return float(Firm.sharedOutputWarehouse[self.address])

    @outputWarehouse.setter
    def outputWarehouse(self, value):
        Firm.sharedOutputWarehouse[self.address] = value

    @property
    def targetAddedValue(self):
        if self.address < 0 or Firm.sharedTargetAddedValue is None:
            return float(self._targetAddedValue)
        return float(Firm.sharedTargetAddedValue[self.address])

    @targetAddedValue.setter
    def targetAddedValue(self, value):
        if self.address < 0 or Firm.sharedTargetAddedValue is None:
            self._targetAddedValue = float(value)
        else:
            Firm.sharedTargetAddedValue[self.address] = value

    def save(self):
        # Ghosts contain only the supplier metadata required on remote ranks.
        # Dynamic warehouse and target values remain in shared memory.
        return (
            self.uid,
            self.sector,
            self.country,
            self.productionType,
            self.address,
        )


def restore_firm(agent_data):
    uid, sector, country, productionType, address = agent_data

    # Ghost Firms are passive supplier handles and never execute Firm behaviour.
    aFirm = Firm(
        uid[0], uid[2], sector, 0, 0.0, 0.0, 0.0, country,
        0.0, 0.0, 0.0, 0.0, 0.0, productionType
    )
    aFirm.address = address
    return aFirm


In [4]:
def statistics(y_true, y_est):

    y_true = np.array(y_true)
    y_est  = np.array(y_est)

    n = y_true.shape[0]
    
    #MAE, Mean Absolute Error
    # $ MAE = \frac{1}{n}\sum_{i=1}^n | yi - \hat{y}_i | $
    mae = abs(y_true - y_est).mean()

    # RMSE, Root Mean Squared Error
    # $ RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^n ( y_i - \hat{y}_i )^2} $
    rmse = (((y_true - y_est)**2).sum()/n)**0.5

    # MAPE, Mean Absolute Percentage Error
    # $ MAPE = \frac{100}{n}\sum_{i=1}^n \left| \frac{y_i - \hat{y}_i}{y_i} \right| $
    safeDiv = np.where(y_true == 0, 0.000001, 0)
    mape = (abs((y_true - y_est) / (y_true+safeDiv)).sum()) * 100 / n

    # R², coefficient of determination
    # $ R^2 = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2} $
    if ((y_true - y_true.mean())**2).sum() == 0: r2 = 0
    else: r2 = 1 - (((y_true - y_est)**2).sum()/((y_true - y_true.mean())**2).sum())**0.5

    return (mae,rmse,mape,r2,n)  #,rmse, mape,r2,count)

  
  
$ MAE = \frac{1}{n}\sum_{i=1}^n | yi - \hat{y}_i | $

$ RMSE = \sqrt{\frac{1}{n}\sum_{i=1}^n ( y_i - \hat{y}_i )^2} $

$ MAPE = \frac{100}{n}\sum_{i=1}^n \left| \frac{y_i - \hat{y}_i}{y_i} \right| $

$ R^2 = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2} $


In [5]:
from IPython.display import display, HTML
def stop_execution(message):
    display(HTML(f"<pre>{message}</pre>"))
    raise SystemExit(0)

In [6]:
def generateOutput(countryLabel, modelOutput, AFF_FirmNumber, countAFF_Firms, propFactor, AFF_WorkerNumber, countAFF_WorkerNumber,
                   Eu_FirmNumber, countActualExPostFirmNumber, Eu_WorkerNumber, countWorkers, Eu_EmployeeCompensations,
                   countEmployeeCompensations, Eu_GDP_withVAT2022, Eu_AddedValue2022, countCapitalR, countActualAddedValue, markupTentative,
                   Eu_Intermediate2022, intermediateTotal, substitutionRate, substitutionRateL, firmSubstitutions,
                   allfirmSubstitutionsByBuyerSector, simulationFirmsExAnteNumber, allfirmSubstitutionsByVendorSector):

    # employeeCompensations_CapitalR_markupAugmented = \
    #   int(markupTentative*(countEmployeeCompensations/1_000_000 + countCapitalR/1000))
    capitalR_fullMarkupAugmented = \
       int(countCapitalR/1000 + (markupTentative-1)*(countEmployeeCompensations/1_000_000 + countCapitalR/1000))
    table_data = [
        [' ',          'EU 27', 'simulation', 'simulation to'],
        [' ',          ' ',     ' ',          'EU scale'],
        ['AFF Firm Number', AFF_FirmNumber, countAFF_Firms, int(countAFF_Firms * propFactor)], 
        ['AFF Worker Number', AFF_WorkerNumber, countAFF_WorkerNumber, int(countAFF_WorkerNumber * propFactor)], 
        ['Firm Number', Eu_FirmNumber, countActualExPostFirmNumber, int(countActualExPostFirmNumber * propFactor)],
        ['Worker Number', Eu_WorkerNumber, countWorkers, int(countWorkers * propFactor)],
        ['Empl. Compens.', int(Eu_EmployeeCompensations), int(countEmployeeCompensations/1_000_000), 
                           int(countEmployeeCompensations * propFactor / 1_000_000)],                                                                                                                              
        ['Cap. R & Op. Surplus', int(Eu_AddedValue2022 - Eu_EmployeeCompensations), capitalR_fullMarkupAugmented, 
                                 int(capitalR_fullMarkupAugmented*propFactor)],
        ['GDP curr. p. (VAT)', Eu_GDP_withVAT2022, 'N.B. contains VAT',' '],
        ['Add. Val. basic p.', Eu_AddedValue2022, int(countActualAddedValue/1_000_000), 
                                # countActualAddedValue was employeeCompensations_CapitalR_markupAugmented in the previous version
                               int((countActualAddedValue/1_000_000)*propFactor)],
        ['Int. Prod. basic p.', int(Eu_Intermediate2022), int(intermediateTotal/1000_000),
         int(intermediateTotal * propFactor / 1000000)]
    ]
    
    print("\n", countryLabel, "\n", file=modelOutput)
    
    print(f"Markup tentative {100*(markupTentative-1)}%", file=modelOutput)
    for row in table_data:
        print("{: >20} {: >20} {: >20} {: >20}".format(*row), file=modelOutput)

    print("\n AFF => Agriculture, Forestry and Fishing\n\n", file=modelOutput)


In [7]:
def generateGDPOutput(countryLabel, modelOutput, propFactor, countActualAddedValue):
                        #countEmployeeCompensations, countCapitalR, markupTentative):
    
    print(countryLabel, int((countActualAddedValue/1_000_000) * propFactor),
          #int(markupTentative*(countEmployeeCompensations * propFactor/1_000_000 + countCapitalR * propFactor/1000)), \
          file=modelOutput)

In [8]:
def match_buyers_to_suppliers(
    buyers,
    suppliers,
    buyer_country: int,
    mult: float,
    rng: np.random.Generator,
    point_mode: str = "systematic",
):
    """
    Matches the buyers of a country with the suppliers of a sector
    through two aligned tapes.

    The buyer does not directly store the reference to the supplier,
    but the supplier's position in the permuted list.

    Parameters
    ----------
    buyers
        List of buyers belonging to the same country.

    suppliers
        List of suppliers belonging to the same sector.

    buyer_country
        Country of the buyers, between 0 and 26.

    mult
        Weight multiplier for suppliers belonging to the same country.

    rng
        NumPy random generator, for example:

        in our case, associate with rng_np of cell 2

    point_mode
        "starts"
            Uses the exact starting point of each buyer's segment.

        "systematic"
            Uses equally spaced points with a common random shift.

        "stratified"
            Uses a random point within each buyer's segment.

        we will use sistematic, as suggested at 
        https://github.com/terna/ejmmp/blob/main/modelEU27/two_tape_method/two_tape_matching_method.pdf

    Returns
    -------
    permuted_suppliers
        List of suppliers in the order of the permuted tape.

    matched_positions
        Integer array. For each buyer, it contains the position
        of the assigned supplier in the permuted_suppliers list.

    permutation
        Array connecting the permuted list to the original list:

        permuted_suppliers[p] is suppliers[permutation[p]]
    """

    n_buyers = len(buyers)
    n_suppliers = len(suppliers)

    if n_buyers == 0:
        return (
            [],
            np.empty(0, dtype=np.int64),
            np.empty(0, dtype=np.int64),
        )

    if n_suppliers == 0:
        raise ValueError("The list of suppliers is empty.")

    if mult <= 0:
        raise ValueError("mult must be strictly positive.")

    if not 0 <= buyer_country <= 26:
        raise ValueError("buyer_country must be between 0 and 26.")

    # ------------------------------------------------------------
    # 1. Extract supplier data
    # ------------------------------------------------------------

    supplier_countries = np.fromiter(
        (supplier.country for supplier in suppliers),
        dtype=np.int16,
        count=n_suppliers,
    )

    added_values = np.fromiter(
        (supplier.targetAddedValue for supplier in suppliers),
        dtype=np.float64,
        count=n_suppliers,
    )

    if np.any(~np.isfinite(added_values)):
        raise ValueError(
            "targetAddedValue contains non-finite values."
        )

    if np.any(added_values < 0):
        raise ValueError(
            "targetAddedValue contains negative values."
        )

    # ------------------------------------------------------------
    # 2. Build the weights for the country-specific version
    #    of tape B
    # ------------------------------------------------------------

    weights = added_values.copy()

    same_country = supplier_countries == buyer_country
    weights[same_country] *= mult

    total_weight = weights.sum()

    if total_weight <= 0:
        raise ValueError(
            "The sum of the suppliers' weights must be positive."
        )

    # ------------------------------------------------------------
    # 3. Shuffle tape B
    # ------------------------------------------------------------

    permutation = rng.permutation(n_suppliers)

    permuted_suppliers = [
        suppliers[original_position]
        for original_position in permutation
    ]

    permuted_weights = weights[permutation]

    # ------------------------------------------------------------
    # 4. Right endpoints of the intervals occupied by suppliers
    # ------------------------------------------------------------

    cumulative_weights = np.cumsum(permuted_weights)

    # Avoid small numerical issues at the final endpoint.
    cumulative_weights[-1] = total_weight

    # ------------------------------------------------------------
    # 5. Buyer positions on tape A
    # ------------------------------------------------------------

    buyer_number = np.arange(
        n_buyers,
        dtype=np.float64,
    )

    if point_mode == "starts":

        buyer_points = (
            buyer_number
            * total_weight
            / n_buyers
        )

    elif point_mode == "systematic":

        # A single random shift for the entire tape A.
        offset = rng.random()

        buyer_points = (
            buyer_number + offset
        ) * total_weight / n_buyers

    elif point_mode == "stratified":

        # A random point within each buyer's segment.
        buyer_points = (
            buyer_number + rng.random(n_buyers)
        ) * total_weight / n_buyers

    else:
        raise ValueError(
            "point_mode must be ‘starts’, "
            "'systematic' or 'stratified'."
        )

    # ------------------------------------------------------------
    # 6. Search for the supplier interval containing each point
    # ------------------------------------------------------------

    matched_positions = np.searchsorted(
        cumulative_weights,
        buyer_points,
        side="right",
    ).astype(np.int64)

    return (
        permuted_suppliers,
        matched_positions,
        permutation,
    )

In [9]:
def printDemandOfferRefill(outAddedV):

    ##### added value int, C, I, and tot
    if outAddedV:
     totalTargetAddedValueBySectIntProduction = [0] * 64
     totalActualAddedValueBySectIntProduction = [0] * 64
     for aFirm in context.agents(agent_type=0):
         if aFirm.productionType == "Int":
             totalTargetAddedValueBySectIntProduction[aFirm.sector - 1] += aFirm.targetAddedValue
             totalActualAddedValueBySectIntProduction[aFirm.sector - 1] += aFirm.actualAddedValue

     print(
     "\nInt. good firms: target and actual added value by sector\n",
     [(i + 1,round(target / 1_000_000, 2),round(actual / 1_000_000, 2),)
         for i, (target, actual) in enumerate(
             zip(totalTargetAddedValueBySectIntProduction, totalActualAddedValueBySectIntProduction))])
     t1 = sum(totalTargetAddedValueBySectIntProduction)
     t2 = sum(totalActualAddedValueBySectIntProduction)
     print("TOTAL:", round(t1 / 1_000_000, 2), round(t2 / 1_000_000, 2))

    
     totalTargetAddedValueBySectCProduction = [0] * 64
     totalActualAddedValueBySectCProduction = [0] * 64
     for aFirm in context.agents(agent_type=0):
         if aFirm.productionType == "C":
             totalTargetAddedValueBySectCProduction[aFirm.sector - 1] += aFirm.targetAddedValue
             totalActualAddedValueBySectCProduction[aFirm.sector - 1] += aFirm.actualAddedValue

     print(
     "\nCons. good firms: target and actual added value by sector\n",
     [(i + 1,round(target / 1_000_000, 2),round(actual / 1_000_000, 2),)
         for i, (target, actual) in enumerate(
             zip(totalTargetAddedValueBySectCProduction, totalActualAddedValueBySectCProduction))])
     t1 = sum(totalTargetAddedValueBySectCProduction)
     t2 = sum(totalActualAddedValueBySectCProduction)
     print("TOTAL:", round(t1 / 1_000_000, 2), round(t2 / 1_000_000, 2))

    
     totalTargetAddedValueBySectIProduction = [0] * 64
     totalActualAddedValueBySectIProduction = [0] * 64
     for aFirm in context.agents(agent_type=0):
         if aFirm.productionType == "I":
             totalTargetAddedValueBySectIProduction[aFirm.sector - 1] += aFirm.targetAddedValue
             totalActualAddedValueBySectIProduction[aFirm.sector - 1] += aFirm.actualAddedValue

     print(
     "\nInv. good firms: target and actual added value by sector\n",
     [(i + 1,round(target / 1_000_000, 2),round(actual / 1_000_000, 2),)
         for i, (target, actual) in enumerate(
             zip(totalTargetAddedValueBySectIProduction, totalActualAddedValueBySectIProduction))])
     t1 = sum(totalTargetAddedValueBySectIProduction)
     t2 = sum(totalActualAddedValueBySectIProduction)
     print("TOTAL:", round(t1 / 1_000_000, 2), round(t2 / 1_000_000, 2))


     totalTargetAddedValueBySectTProduction = [0] * 64
     totalActualAddedValueBySectTProduction = [0] * 64
     for aFirm in context.agents(agent_type=0):
         
         totalTargetAddedValueBySectTProduction[aFirm.sector - 1] += aFirm.targetAddedValue
         totalActualAddedValueBySectTProduction[aFirm.sector - 1] += aFirm.actualAddedValue

     print(
     "\nAll good firms: target and actual added value by sector\n",
     [(i + 1,round(target / 1_000_000, 2),round(actual / 1_000_000, 2),)
         for i, (target, actual) in enumerate(
             zip(totalTargetAddedValueBySectTProduction, totalActualAddedValueBySectTProduction))])
     t1 = sum(totalTargetAddedValueBySectTProduction)
     t2 = sum(totalActualAddedValueBySectTProduction)
     print("TOTAL:", round(t1 / 1_000_000, 2), round(t2 / 1_000_000, 2))

    ##### demand of intermediate products by firms of each speficic sector, any productionType (Naio cols)
    
    totalIntermediateWarehouseRefillByUsingSect = [0] * 64
    for aFirm in context.agents(agent_type=0):
        for i in range(64):
            totalIntermediateWarehouseRefillByUsingSect[aFirm.sector-1] += aFirm.intermediateWarehouseRefill[i]

    print(
        "\nAll firms refill needs: demand of int. goods by using sector, any productionType (sum Naio col)\n",
        [(i + 1, round(value / 1_000_000, 2))
         for i, value in enumerate(totalIntermediateWarehouseRefillByUsingSect)]
    )
    t1 = sum(totalIntermediateWarehouseRefillByUsingSect)
    print("TOTAL:", round(t1 / 1_000_000, 2))

    ##### demand of intermediate products by all the producing sector from each speficic sector (Naio rows)   

    totalIntermediateWarehouseRefillByProducingSect = [0] * 64
    for aFirm in context.agents(agent_type=0):
        for i in range(64):
            totalIntermediateWarehouseRefillByProducingSect[i] += aFirm.intermediateWarehouseRefill[i]

    print(
        "\nAll firms refill needs: demand of int. goods from a specific sector by all producing sector (sum Naio row)\n",
        [(i + 1, round(value / 1_000_000, 2))
         for i, value in enumerate(totalIntermediateWarehouseRefillByProducingSect)]
    )
    t1 = sum(totalIntermediateWarehouseRefillByProducingSect)
    print("TOTAL:", round(t1 / 1_000_000, 2))


    
    # offer of int goods 
    totalOutputWarehousesOfIntermediateGoods = [0] * 64
    for aFirm in context.agents(agent_type=0):
        if aFirm.productionType == "Int":
            totalOutputWarehousesOfIntermediateGoods[aFirm.sector - 1] += aFirm.outputWarehouse

    print(
        "\nInt good firms output warehouses by sector\n",
        [(i + 1, round(value / 1_000_000, 2))
         for i, value in enumerate(totalOutputWarehousesOfIntermediateGoods)]
    )
    t2 = sum(totalOutputWarehousesOfIntermediateGoods)
    print("TOTAL:", round(t2 / 1_000_000, 2))


    print(
        "\nDifferences\n",
        [(i + 1, round((a - b) / 1_000_000, 2))
         for i, (a, b) in enumerate(zip(
             totalOutputWarehousesOfIntermediateGoods,
             totalIntermediateWarehouseRefillByProducingSect
         ))]
    )
    print("TOTAL DIFF:", round((t2 - t1) / 1_000_000, 2))

In [10]:
def scalingWorkerDistribution(worker_distribution):
    values = [0] * 1849
    # class 1-9
    tot = 0
    for i in range(9): 
        tot += worker_distribution[i]
    for i in range(9):
        values[i] = worker_distribution[i] / tot
    # class 10-19
    tot = 0 
    for i in range(9, 19):
        tot += worker_distribution[i]
    for i in range(9, 19):
        values[i] = worker_distribution[i] / tot
    # class 20-49
    tot = 0 
    for i in range(19, 49):
        tot += worker_distribution[i]
    for i in range(19, 49):
        values[i] = worker_distribution[i] / tot
    # class 50-249
    tot = 0 
    for i in range(49, 249):
        tot += worker_distribution[i]
    for i in range(49, 249):
        values[i] = worker_distribution[i] / tot
    # class 250 or more (1849)
    tot = 0 
    for i in range(249, 1849):
        tot += worker_distribution[i]
    for i in range(249, 1849):
        values[i] = worker_distribution[i] / tot     
    return values



def buildAtomicLockHelper(sharedComm):

    # This tiny C11 helper performs CPU atomic operations directly on
    # the shared-memory lock array. No MPI operation occurs per purchase.
    helperDir = Path(tempfile.gettempdir()) / "modelEU27_atomic_lock"
    helperDir.mkdir(parents=True, exist_ok=True)

    sourcePath = helperDir / "atomic_lock.c"
    libraryPath = helperDir / (
        "libatomic_lock.dylib"
        if platform.system() == "Darwin"
        else "libatomic_lock.so"
    )

    source = """
#include <stdint.h>
#include <stdatomic.h>

int acquire_lock(int32_t *lock)
{
    int32_t expected;
    for (;;) {
        expected = 0;
        if (atomic_compare_exchange_weak_explicit(
                (_Atomic int32_t *) lock,
                &expected,
                1,
                memory_order_acquire,
                memory_order_relaxed)) {
            return 1;
        }
    }
}

void release_lock(int32_t *lock)
{
    atomic_store_explicit(
        (_Atomic int32_t *) lock,
        0,
        memory_order_release
    );
}
"""

    if sharedComm.Get_rank() == 0:
        sourcePath.write_text(source, encoding="utf-8")

        if platform.system() == "Darwin":
            command = [
                "cc", "-O3", "-std=c11", "-dynamiclib",
                str(sourcePath), "-o", str(libraryPath),
            ]
        else:
            command = [
                "cc", "-O3", "-std=c11", "-shared", "-fPIC",
                str(sourcePath), "-o", str(libraryPath),
            ]

        subprocess.run(
            command,
            check=True,
            capture_output=True,
            text=True,
        )

    sharedComm.Barrier()

    library = ctypes.CDLL(str(libraryPath))
    library.acquire_lock.argtypes = [ctypes.POINTER(ctypes.c_int32)]
    library.acquire_lock.restype = ctypes.c_int
    library.release_lock.argtypes = [ctypes.POINTER(ctypes.c_int32)]
    library.release_lock.restype = None

    return library


class Model:
    
    def __init__(self):
        

        self.count_months = -1
        self.propFactor = 0

        # Firm production types to ghost on all the other ranks.
        # Extend this set if additional production types become remote suppliers.
        self.ghostProductionTypes = {"Int"}

        # Detailed timing diagnostics can be disabled from modelEU27.yaml.
        self.detailedTiming = not params.get('stop_detailed_timing', False)
        if self.detailedTiming:
            self.phaseTimingWall = {}
            self.phaseTimingCpu = {}
            self.reduceTimingWall = {}
            self.reduceTimingCpu = {}
        
        ##################################### SCHEDULE ########################################
        #######################################################################################
        runner.schedule_event(          0.0,     self.createFirmSubsets)
        if self.detailedTiming:
            runner.schedule_repeating_event(0.1,  1, self.timedSupply)
            runner.schedule_repeating_event(0.2,  1, self.timedDemand)
            runner.schedule_repeating_event(0.3,  1, self.timedAggregatingResultsByCountry)
            runner.schedule_repeating_event(0.35, 1, self.timedSelectSuppliers)
            runner.schedule_repeating_event(0.4,  1, self.timedAggregatingRanks)
        else:
            runner.schedule_repeating_event(0.1,  1, self.supply)
            runner.schedule_repeating_event(0.2,  1, self.demand)
            runner.schedule_repeating_event(0.3,  1, self.aggregatingResultsByCountry)
            runner.schedule_repeating_event(0.35, 1, self.selectSuppliers)
            runner.schedule_repeating_event(0.4,  1, self.aggregatingRanks)
        runner.schedule_repeating_event(0.7,  1, self.showResults)

        #runner.schedule_stop(0.8) # N x 12 identifies the number of years to be run as one time-step is one month
        runner.schedule_stop(params['howManyCycles']) # N x 12 identifies the number of years to be run 
                                                      # as one time-step is one month
        runner.schedule_end_event(self.finish)

        self.eu_countries = ['Austria', 'Belgium', 'Bulgaria', 'Croatia', 'Cyprus', 'Czechia', 
                             'Denmark', 'Estonia', 'Finland', 'France', 'Germany', 'Greece', 
                             'Hungary', 'Ireland', 'Italy', 'Latvia', 'Lithuania', 'Luxembourg', 
                             'Malta', 'Netherlands', 'Poland', 'Portugal', 'Romania', 'Slovakia', 
                             'Slovenia', 'Spain', 'Sweden', 'totEU27']
        country_numid_list = list(range(27))

        self.selected_countries = params['selected_countries']
    
        eu_productivities = pd.read_pickle("./eu_productivities.xp") # generated in computing_productivity.ipynb
        self.adjustingCoefficient = eu_productivities['no agri tot prod Italy-adjusted'] # adjustment index, based on Istat data on capital 
        self.agri_adjustingCoefficient = eu_productivities['agri tot prod']  
        nama_av = pd.read_pickle("./nama_av.xp") # generated in computing_productivity.ipynb
        self.shareConsumptionDemandTable = pd.read_pickle("./shareConsumptionDemand.xp")  # generated in initialConsumptionDemand.ipynb

        self.lc_countries = pd.read_pickle("./lc_countries.xp") # generated in lc_countries.ipynb
        # lc_countries is a column of the labor cost index by country
        
        self.gdp27_comparison = pd.read_pickle("./gdp27_comparison.xp") # generated in GNP27basicPrices.ipynb
        # at the moment here only for comparison reasons (should not be relevant to new steps of the model)
        
        self.consumptionShares = pd.read_pickle("./shareConsumptionDemand.xp") # generated in initialConsumptionDemand.ipynb

        #print(self.consumptionShares.iloc[2,-2],self.consumptionShares.iloc[2,-1])
        #self.consumptionShares = self.consumptionShares.set_index("country")
        #print(self.consumptionShares.at["Bulgaria", "private_cons / gdp"], self.consumptionShares.at["Bulgaria", "gov_cons /gdp"])

        
        ####################################################################################################
        ###################################### CREATE FIRM AGENTS ##########################################
        ####################################################################################################

        with open('naio_io_N.xp', 'rb') as f:
            intermediateInputs_float64 = pickle.load(f) #numpy array, only numeric, see last row of firm-features-generation.ipynb
                                                #eliminating nan in col 43 (used), not in 64 (never used)
        self.intermediateInputs = intermediateInputs_float64.astype(float).tolist()
                                                # from now onwards we have to call its values by using * [i][j] * as indexes
        
        #firm investment shares
        nama = params['tableOfInvestmentShares']
        print('rank', rank, 'tableOfInvestmentShare', nama)

        if nama==0: self.invShares = pd.read_pickle("./invTableNoNama.xp") #ignoring nama infos
        if nama==1: self.invShares = pd.read_pickle("./invTableNama.xp")   #using nama infos
        if nama==2: self.invShares = pd.read_pickle("./invTableNamaIPF.xp")   #using nama infos

        self.invSharesNp = self.invShares.to_numpy()
        #print(self.invSharesNp)

        #65 rows x 66 columns
        #print (self.invShares)
        #print (self.invShares.iloc[0,0]) #Crop
        #print (self.invShares.iloc[1,0]) #Forest
        #print (self.invShares.iloc[0,1]) #0.002242 values for invTableNoNama (here and following)
        #print (self.invShares.iloc[1,1]) #0.002242
        #print (self.invShares.iloc[0,2]) #0.000131
        #print (self.invShares.iloc[1,2]) #0.000131
        #print (self.invShares.iloc[43,0]) #Imputed 
        #print (self.invShares.iloc[44,0]) #Real
            
        for i in range(64): #64, as rows 0:63
            self.intermediateInputs[i][43]=0
            #self.intermediateInputs[i][64]=0 #col 64 never used, as sector 65 was dropped

        fileName = "ff_with_class_limits.csv" #input("file name? ")

        # how many firms
        #self.simulationFirmsExAnteNumber = int(input("how many firms? "))
        self.simulationFirmsExAnteNumber = params['howManyFirms']
        print('rank', rank, 'how many firms', self.simulationFirmsExAnteNumber)

        with open("workers_distribution.xp", "rb") as f:
            worker_distribution = pickle.load(f)

        scaled_worker_distribution = scalingWorkerDistribution(worker_distribution) 
        L  = [(0,9), (9,19), (19,49), (49,249), (249,1849)] #probabilities
        #scaled_worker_distribution[L[cl][0]:L[cl][1]] to call specific values within a class
        wN = [i for i in range(1,1850)]
        #wN[L[cl][0]:L[cl][1]] to call specific values within a class
        
        # smart capital
        """
        self.smart_capital = 2
        while self.smart_capital != 0 and self.smart_capital != 1:
            self.smart_capital = int(input("use smart capital? (Please, use only 1|0 to express True/False values) "))
            if self.smart_capital != 0 and self.smart_capital != 1: print("Please, use only 1|0 to express True/False values")
        """
        self.smart_capital = params['useSmartCapital']
        print('rank', rank, 'use smart capital', self.smart_capital)

        # from addedvalue_by_country.ipynb - in folder structuralData
        with open('addedvalue_countryshares.xp', 'rb') as f:
            addedvalue_countryshares = pickle.load(f)
            countryshare_list = addedvalue_countryshares['Share in 2020'].tolist()
            # addedvalue_countryshares['Country', 'Share in 2020'] where 'Country' is an index, not a column
            # to call the share use addedvalue_countryshares.iloc[0,x]
            # can also call by country doing addedvalue_countryshares.at['Austria', 'Share in 2020']

        # from sbs_countries.ipynb
        eu_firms_by_employed_number = pd.read_pickle("./eu_firms_by_employed_number.xp") # this is a list of dfs (0-9, 10-19,20-49,50-249,>250)
        sector_special_cases = ['1', '2', '3', '44'] # agri-silvi-fish and imputed rents
        agri_eu27 = pd.read_pickle("./agri_eu27base.xp") # agri-silvi-fish shares (countries are cols to keep 
                                                        # consistency with eu_firms_by_employed_number)
                                                       # the agri_eu27base file comes from ./special_sectors_eu27.ipynb (cell 7)
        agri_eu27.index = ["Agricultural share", "Forestry share", "Fishing share"]

        prob_country_agri = agri_eu27.loc["Agricultural share"].tolist()
        prob_country_silvi = agri_eu27.loc["Forestry share"].tolist()
        prob_country_fishing = agri_eu27.loc["Fishing share"].tolist()

        self.eu27_owners = pd.read_pickle("./eu27_ownership.xp") # 44 sector, imputed rents from houses ownership by EU country
                                                                 # generated from special_sectors_EU27.ipynb

        #importing csv file containing info about firms 
        with open(fileName, newline='') as csvfile:
            firmReader= csv.reader(csvfile, delimiter=',')#, quoting=csv.QUOTE_NONNUMERIC)
    
            self.rowNumber=-1 #to skip the row of the headers
            k=0 
            createImputedRentsFirms = True
            if rank != 0: createImputedRentsFirms = False # to avoid firms creation from sect44 in ranks != 0
            
            for row in firmReader: #for each record in .csv
                #print(row)
                labor = 0 
                if self.rowNumber >= 0 and tab924[self.rowNumber][rank] == 1: # skip if self.rowNumber == -1 or tab924[][] == 0
                    if row[4]=='': row[4]=0 # it pertains with last rows (naio sector 65)                       
                    sector = row[0]

                    #*****************************************************************************************
                    initialFirmCreationNumber= round(float(row[4]) * self.simulationFirmsExAnteNumber)
                    firmCreationNumber = initialFirmCreationNumber 
                    
                    if sector =='44' and createImputedRentsFirms:
                        firmCreationNumber= 27 # number of EU countries
                        createImputedRentsFirms = False

                    countImputedRentsFirms = 0

                    #*****************************************************************************************
                    # class attribution
                    if row[3] == "From 0 to 9 persons employed":    cl = 0
                    if row[3] == "From 10 to 19 persons employed":  cl = 1
                    if row[3] == "From 20 to 49 persons employed":  cl = 2
                    if row[3] == "From 50 to 249 persons employed": cl = 3
                    if row[3] == "250 persons employed or more":    cl = 4

                            
                    for i in range(firmCreationNumber):
                        if row[0] == '1' or row[0] == '2' or row[0] == '3': #agri, silvi and fishing
                            randomizer = rng.uniform()
                            if randomizer <= 0.2: labor = 0
                            elif randomizer <= 0.9 : labor = 1
                            else: labor = 2
                        else:
                            # use L and scaled_worker_distribution
                            labor = rng_np.choice(wN[L[cl][0]:L[cl][1]], p=scaled_worker_distribution[L[cl][0]:L[cl][1]])
                            
                            if sector =='44': labor = 0 # no employees in a virtual sector
                        
                        
                        # assignation of EU country label/number 
                        country_randomizer = rng.uniform()
                        # assigning rule: we attribute the first country with a cumulative share higher than the random number we draw
                        # non-special cases sectors come from the sbs_countries file, while agriculture sector is treaten in the agri_eu27 file
                        country_counter = 0
                        #if not row[0] in sector_special_cases:
                        if not row[0] in sector_special_cases and int(row[0]) != 61 and int(row[0]) != 64:
                            #if eu_firms_by_employed_number[cl].iloc[int(row[0]), -1] != 0: # to avoid sectors with all zeros 
                                #while eu_firms_by_employed_number[cl].iloc[int(row[0]), country_counter] <= country_randomizer: country_counter += 1
                            prob_country_sec_cl = eu_firms_by_employed_number[cl].loc[int(row[0])].squeeze().tolist()

                            country_counter = rng_np.choice(country_numid_list, p=prob_country_sec_cl)  
                         
                        
                        elif row[0] == '44' and rank == 0: # imputed rents
                            country_counter = countImputedRentsFirms
                            countImputedRentsFirms += 1

                        elif int(row[0]) == 61 or int(row[0]) == 64: # 61: membership organizations, 64: services by households
                            country_counter = rng_np.choice(country_numid_list, p=countryshare_list) 
                            
                                
                        else: # agri forestry and fishing
                            if int(row[0]) == 1: country_counter = rng_np.choice(country_numid_list, p=prob_country_agri)
                            if int(row[0]) == 2: country_counter = rng_np.choice(country_numid_list, p=prob_country_silvi)
                            if int(row[0]) == 3: country_counter = rng_np.choice(country_numid_list, p=prob_country_fishing)

                   
                        capital= float(row[11]) + rng.random()*(float(row[12]) - float(row[11]))
                        if self.smart_capital:
                            capital = labor * float(row[10]) # row[10] = recipe 
                            
                        capitalR = float(row[13])
                        wage = float(row[14])

                        # adjusting wage and capital accounting for differences in national productivity
                        # the adjustingCoefficient is a vector containing indeces of national productivity (for agri and no-agri sectors)
                                      # for all the countries (ordered in the standard order in which we report the list of EU countries)
                        
                        if row[0] == '1' or row[0] == '2' or row[0] == '3': #agri, silvi and fishing
                            wage *= self.agri_adjustingCoefficient[country_counter]
                            capital *= self.agri_adjustingCoefficient[country_counter]
                        else:
                            #wage *= self.adjustingCoefficient[country_counter]
                            wage *= self.lc_countries[country_counter]
                            capital *= self.adjustingCoefficient[country_counter]
                        
                        
                        # !!!
                        # for the intermediate inputs each firm must grabble its values directly from the self.intermediateInputs table
                        # by using self.intermediateInputs address which is internal to the model

                        recipe = float(row[10])
                        countryRecipe = recipe * self.adjustingCoefficient[country_counter]
                        
                        if row[0] == '1' or row[0] == '2' or row[0] == '3': 
                            laborProductivity = nama_av['agri_tot_prod'][country_counter]
                        else: 
                            laborProductivity = nama_av['noagri_tot_prod'][country_counter]
                         
                        avgAssetsUsefulLife = 12*12 # 12 years, now supposed to be equal for all the sectors (except real estate sector)
                        avgPlannedMarkup = params['plannedMarkup']
                        productionType = row[2]
                        #orderObservationFrequency=rng.integers(row[12], row[13]+1)
                        
                        ################################# adding aFirm ##################################
                        #################################################################################

                        # sector 1-64
                        # country_counter 0-26
                        aFirm =Firm(k, rank, int(sector), labor, capital, capitalR, wage, country_counter,\
                                    recipe, countryRecipe, laborProductivity, avgAssetsUsefulLife, avgPlannedMarkup, productionType) 
                                    #orderObservationFrequency
                        # here we pass "self" as the address of the model to provide all firms with the self.intermediateInputs table
                        
                        context.add(aFirm)
                        k += 1 # first element of the UID of the agents
                        #print(aFirm.uid, aFirm.sector, aFirm.country)

                        #memo: in firm _init_ we have estimatedAddedValueTminus1
                        # Creating t=0 intermediate warehouse using estimated added value at t-1
                        for j in range(64):
                            aFirm.intermediateWarehouse[j] = self.intermediateInputs[j][aFirm.sector-1] * aFirm.estimatedAddedValueTminus1
             
                        
                self.rowNumber += 1
                self.simulationFirmsExPostNumber=k #one more, here is a count, not an id
                
            print(self.simulationFirmsExPostNumber, "firms created in rank ", rank)

            


        
        self.countActualExPostFirmNumber = len(list(context.agents(agent_type=0)))

        # Shared memory for Firm transactional state.
        self.firmCounts = comm.allgather(self.countActualExPostFirmNumber)
        self.firmOffsets = [0] * rankNum
        for i in range(1, rankNum):
            self.firmOffsets[i] = self.firmOffsets[i-1] + self.firmCounts[i-1]
        self.totalFirmNumber = sum(self.firmCounts)

        self.sharedComm = comm.Split_type(MPI.COMM_TYPE_SHARED, key=rank)
        self.sharedRank = self.sharedComm.Get_rank()
        self.sharedRankNum = self.sharedComm.Get_size()
        if self.sharedRankNum != rankNum:
            raise RuntimeError('All ranks must run on the same shared-memory node')

        floatSize = np.dtype(np.float64).itemsize
        intSize = np.dtype(np.int32).itemsize

        if self.sharedRank == 0:
            outputWarehouseBytes = self.totalFirmNumber * floatSize
            targetAddedValueBytes = self.totalFirmNumber * floatSize
            supplierLockBytes = self.totalFirmNumber * intSize
        else:
            outputWarehouseBytes = 0
            targetAddedValueBytes = 0
            supplierLockBytes = 0

        self.outputWarehouseWin = MPI.Win.Allocate_shared(outputWarehouseBytes, floatSize, comm=self.sharedComm)
        outputWarehouseBuffer, _ = self.outputWarehouseWin.Shared_query(0)
        Firm.sharedOutputWarehouse = np.ndarray(buffer=outputWarehouseBuffer, dtype=np.float64,
                                                shape=(self.totalFirmNumber,))

        self.targetAddedValueWin = MPI.Win.Allocate_shared(targetAddedValueBytes, floatSize, comm=self.sharedComm)
        targetAddedValueBuffer, _ = self.targetAddedValueWin.Shared_query(0)
        Firm.sharedTargetAddedValue = np.ndarray(buffer=targetAddedValueBuffer, dtype=np.float64,
                                                 shape=(self.totalFirmNumber,))

        # One int32 lock per Firm, indexed by the global Firm address.
        self.supplierLockWin = MPI.Win.Allocate_shared(
            supplierLockBytes,
            intSize,
            comm=self.sharedComm,
        )

        supplierLockBuffer, _ = self.supplierLockWin.Shared_query(0)

        self.supplierLocks = np.ndarray(
            buffer=supplierLockBuffer,
            dtype=np.int32,
            shape=(self.totalFirmNumber,),
        )

        if self.sharedRank == 0:
            Firm.sharedOutputWarehouse[:] = 0.0
            Firm.sharedTargetAddedValue[:] = 0.0
            self.supplierLocks[:] = 0

        self.sharedComm.Barrier()

        # Compile and load the C11 atomic helper once on this node.
        self.atomicLockLibrary = buildAtomicLockHelper(
            self.sharedComm
        )

        # Assign every local Firm its stable global address before ghosts
        # are created and before shared Firm state is accessed.
        for aFirm in context.agents(agent_type=0):
            aFirm.address = (
                self.firmOffsets[rank]
                + aFirm.uid[0]
            )

            # Copy the pre-shared target value into the Firm's global slot.
            Firm.sharedTargetAddedValue[
                aFirm.address
            ] = aFirm._targetAddedValue

        self.sharedComm.Barrier()

        # Request configured Firm production types from every other rank.
        # Extending ghostProductionTypes is sufficient to ghost more types.
        localGhostCandidates = [
            aFirm.uid
            for aFirm in context.agents(agent_type=0)
            if aFirm.productionType in self.ghostProductionTypes
        ]

        ghostCandidatesByRank = comm.allgather(localGhostCandidates)

        requestedFirms = []
        for sourceRank, sourceCandidates in enumerate(ghostCandidatesByRank):
            if sourceRank == rank:
                continue

            for uid in sourceCandidates:
                requestedFirms.append((uid, sourceRank))

        self.firmGhosts = context.request_agents(
            requestedFirms,
            restore_firm,
        )

        self.AFF_FirmNumber = 10000000
        self.Eu_FirmNumber = 32251874 + self.AFF_FirmNumber
        self.AFF_WorkerNumber = 9000000
        self.Eu_WorkerNumber = 162467679 + 9000000
        self.Eu_EmployeeCompensations = 7447036.79
        self.Eu_GDP_withVAT2022 = 16144780 #https://ec.europa.eu/eurostat/databrowser/view/tec00001/default/table?lang=en
        self.Eu_AddedValue2022 = 14303899 #naio table 2022 (milions) total Added value, gross
        self.Eu_Intermediate2022 = 16939701.18 	 
        if self.propFactor == 0: self.propFactor = self.Eu_FirmNumber / self.countActualExPostFirmNumber
        print("propFactor", self.propFactor,"in rank",rank)

        
    ###### firm subsets ######
    # Timing wrappers for the main monthly phases.
    def timedCall(self, phaseName, method):
        wallStart = time.perf_counter()
        cpuStart = time.process_time()
        method()
        self.phaseTimingWall[phaseName] = time.perf_counter() - wallStart
        self.phaseTimingCpu[phaseName] = time.process_time() - cpuStart


    def timedSupply(self):
        self.timedCall("supply", self.supply)


    def timedDemand(self):
        self.timedCall("demand", self.demand)


    def timedAggregatingResultsByCountry(self):
        self.timedCall("country", self.aggregatingResultsByCountry)


    def timedSelectSuppliers(self):
        self.timedCall("suppliers", self.selectSuppliers)


    def timedAggregatingRanks(self):
        self.reduceTimingWall = {}
        self.reduceTimingCpu = {}
        self.timedCall("ranks", self.aggregatingRanks)

        print(
            f"TIMING rank {rank:3d} month {self.count_months + 1:3d} | "
            f"wall: supply {self.phaseTimingWall.get('supply', 0.0):.6f}  "
            f"demand {self.phaseTimingWall.get('demand', 0.0):.6f}  "
            f"country {self.phaseTimingWall.get('country', 0.0):.6f}  "
            f"suppliers {self.phaseTimingWall.get('suppliers', 0.0):.6f}  "
            f"ranks {self.phaseTimingWall.get('ranks', 0.0):.6f}  "
            f"pack {self.reduceTimingWall.get('pack', 0.0):.6f}  "
            f"reduce {self.reduceTimingWall.get('reduce', 0.0):.6f}  "
            f"unpack {self.reduceTimingWall.get('unpack', 0.0):.6f} | "
            f"cpu: supply {self.phaseTimingCpu.get('supply', 0.0):.6f}  "
            f"demand {self.phaseTimingCpu.get('demand', 0.0):.6f}  "
            f"country {self.phaseTimingCpu.get('country', 0.0):.6f}  "
            f"suppliers {self.phaseTimingCpu.get('suppliers', 0.0):.6f}  "
            f"ranks {self.phaseTimingCpu.get('ranks', 0.0):.6f}  "
            f"pack {self.reduceTimingCpu.get('pack', 0.0):.6f}  "
            f"reduce {self.reduceTimingCpu.get('reduce', 0.0):.6f}  "
            f"unpack {self.reduceTimingCpu.get('unpack', 0.0):.6f}",
            flush=True,
        )


    def createFirmSubsets(self):
        
        # creating all firms database by sector, one global and one for each productionType
        self.firmListBySector =  [[] for _ in range(64)]  
        self.firmListBySectorWithProductionTypeC   =  [[] for _ in range(64)]  
        self.firmListBySectorWithProductionTypeI   =  [[] for _ in range(64)]  
        self.firmListBySectorWithProductionTypeInt =  [[] for _ in range(64)]  

        for aFirm in context.agents(agent_type=0):
            self.firmListBySector[aFirm.sector-1].append(aFirm)
            if aFirm.productionType =="C":   self.firmListBySectorWithProductionTypeC[aFirm.sector-1].append(aFirm)
            if aFirm.productionType =="I":   self.firmListBySectorWithProductionTypeI[aFirm.sector-1].append(aFirm)
            if aFirm.productionType =="Int": self.firmListBySectorWithProductionTypeInt[aFirm.sector-1].append(aFirm)

        # creating all firms database by country
        self.firmListByCountry =  [[] for _ in range(27)]

        for aFirm in context.agents(agent_type=0):
            self.firmListByCountry[aFirm.country].append(aFirm) 

    
        # Ghosts are passive and are added only to supplier-side subsets.
        for aFirm in self.firmGhosts:
            self.firmListBySector[aFirm.sector-1].append(aFirm)
            if aFirm.productionType == "C":   self.firmListBySectorWithProductionTypeC[aFirm.sector-1].append(aFirm)
            if aFirm.productionType == "I":   self.firmListBySectorWithProductionTypeI[aFirm.sector-1].append(aFirm)
            if aFirm.productionType == "Int": self.firmListBySectorWithProductionTypeInt[aFirm.sector-1].append(aFirm)


    # Firm ghosts have already been created in __init__.
    # supply 
    def supply(self):

        self.count_months += 1
        
        # here we use 28 but the last positiion of the vector can be an empty space 
        
        self.countWorkers = [0] * 28
        self.countEmployeeCompensations = [0] * 28
        self.countCapitalR = [0] * 28
        self.countActualAddedValue = [0] * 28
        self.countAFF_Firms = [0] * 28
        self.countAFF_WorkerNumber = [0] * 28
        self.intermediateTotalSupplyBySector = [0] * 64
        self.intermediateTotalSupplyByCountry = [0] * 28
        self.intermediateTotalSupplyBySectorAndCountry =  [[0] * 28 for _ in range(64)]
        self.intBySectors=[0]*64
        self.addValBySectors=[0]*64
        self.markupTentative= params['plannedMarkup']


        for aFirm in context.agents(agent_type=0):
      
        ###################################################################################################
        ########################### COMPUTING ADDED VALUE BY FIRM IN EACH RANK ############################
        ###################################################################################################
            
            self.countWorkers[aFirm.country] += aFirm.labor
            self.countEmployeeCompensations[aFirm.country] += (aFirm.labor*aFirm.wage) / 12
            self.countCapitalR[aFirm.country] += (aFirm.capital*aFirm.capitalR) / 12
            if int(aFirm.sector) <= 3: 
                self.countAFF_Firms[aFirm.country] += 1
                self.countAFF_WorkerNumber[aFirm.country] +=aFirm.labor
                
            aFirm.targetAddedValue= float(aFirm.labor * aFirm.wage + aFirm.capital * 1000 * aFirm.capitalR) / 12   # float to avoid np.float64 in vector
            aFirm.targetAddedValue *= self.markupTentative 

            
    
            # firms account for the availability of EACH intermediate good in the intermediate warehouse to produce
            # we have to consider the quantity (in value) that we need for each intermediate good to produce our desired added value
            
            # if this quantity is not available (or partially not available) the firm will limit the production to the quantity 
                                                                                              # of intermediate goods available
            # for simiplicity, we consider for this purpose only intermediate goods evaluated at least at 1% of the firm added value
                
            # notice that firms usually use multiple intermediate goods
            # if many of them are unavailable we have to consider the lower quantity available for the firm production in that cycle 
            # worst case scenario q of one intermediate good = 0 ---> production = 0
        
            
            aFirm.actualAddedValue = aFirm.targetAddedValue
            for j in range(64):
            # what is doing: cutting actualAddedValue
                if aFirm.intermediateWarehouse[j] < (aFirm.actualAddedValue * self.intermediateInputs[j][aFirm.sector-1])* 0.99\
                                                and self.intermediateInputs[j][aFirm.sector-1] > 0.01:
                    aFirm.actualAddedValue *= aFirm.intermediateWarehouse[j]/ (aFirm.actualAddedValue * self.intermediateInputs[j][aFirm.sector-1])
            
            # after determining aFirm.actualAddedValue across all 64 sectors we can withdraw the corresponding 
                                                        # intermediate goods quotas from their warehouse

            #if aFirm.targetAddedValue - aFirm.actualAddedValue > 0.001: print(aFirm.uid, aFirm.sector-1)


            # sum of all the actual added values
            self.countActualAddedValue[aFirm.country] += aFirm.actualAddedValue

            intermediateWarehouseWithdraw = 0
            for j in range(64):
                aFirm.intermediateWarehouse[j] -= (aFirm.actualAddedValue * self.intermediateInputs[j][aFirm.sector-1])
                intermediateWarehouseWithdraw += (aFirm.actualAddedValue * self.intermediateInputs[j][aFirm.sector-1])
            
            ############################ intermediate goods firm production ###############################
            if aFirm.productionType == "Int": 
                # totalizers to measure this quantity in a matrix sector x country
                self.intermediateTotalSupplyBySector[aFirm.sector-1] +=  aFirm.actualAddedValue + intermediateWarehouseWithdraw
                self.intermediateTotalSupplyByCountry[aFirm.country] +=  aFirm.actualAddedValue + intermediateWarehouseWithdraw
                self.intermediateTotalSupplyBySectorAndCountry[j][aFirm.country] += aFirm.actualAddedValue + intermediateWarehouseWithdraw


        # should include: a) direct imports from buyers from those sectors, b) the valorization of intermediate products ???
        
        ###################################################################################################
        ###################### SENDING PRODUCTS TO WAREHOUSES BY FIRM IN EACH RANK ########################
        ###################################################################################################
            aFirm.outputWarehouse += aFirm.actualAddedValue + intermediateWarehouseWithdraw 
            # must include a) intermediate goods employed in production; b) imports by producing firm/sector
            
        #print("Supply", self.intermediateTotalSupplyBySector)                
        


    def demand(self): 

        self.firmSubstitutions= [0] * 28
        self.substitutionRate =0.1
        self.substitutionRateL=0.01667 #for sectors 44 and 45, 60 years of duration
        #self.allfirmSubstitutionsByVendorSector=[0]*65
        self.allfirmSubstitutionsByVendorSector=np.zeros([65,28])
        self.allfirmSubstitutionsByBuyerSector=np.zeros([65,28])
        self.intermediateTotalDemandByCountry = [0] * 28
        self.intermediateTotalDemandBySector = [0] * 64
        self.intermediateTotalDemandBySectorAndCountry =  [[0] * 28 for _ in range(64)]
        
        for aFirm in context.agents(agent_type=0):
    
        ###################################################################################################
        ############### CAPITAL SUBSTITUIONS GOODS PURCHASING ORDERS BY FIRM IN EACH RANK #################
        ###################################################################################################
        # here what matters is the substitutions by buyer, whereas those by vendor are relevant only for statistical purposes for check
        
            # imputed rents (special case)
            if aFirm.sector == 44: 
                # from ff_with_class_limits, we use the number of the sector not the position!
                aFirm.capital= (30_000_000 * 1000 * self.eu27_owners.iloc[-1,aFirm.country]) / 12
                
                # the whole EU capital of the sector, to be subdivided by countries
                # see structuralData > firm-features-generation for explanation, cell "Imputed rents special case"
                self.firmSubstitutions[aFirm.country] += ( ((aFirm.capital*self.substitutionRateL)/self.Eu_FirmNumber)\
                                                           *self.simulationFirmsExAnteNumber) / 12
                                        # it will be reported to the EU scale
                    
                self.allfirmSubstitutionsByBuyerSector[aFirm.sector-1, aFirm.country] += ( ((aFirm.capital*self.substitutionRateL)\
                                                                         /self.Eu_FirmNumber)*self.simulationFirmsExAnteNumber) / 12

                # buying uniquely from constructions (ie sector 27!)
                self.allfirmSubstitutionsByVendorSector[27-1, aFirm.country]+= ( (aFirm.capital*self.substitutionRateL/self.Eu_FirmNumber)*\
                                                                                                  self.simulationFirmsExAnteNumber) / 12

            # real estate (special case 2)
            elif aFirm.sector == 45: # from ff_with_class_limits, we use the number of sector not the position!
                self.firmSubstitutions[aFirm.country] +=  (aFirm.capital*self.substitutionRateL) / 12
                self.allfirmSubstitutionsByBuyerSector[aFirm.sector-1, aFirm.country] += (aFirm.capital*self.substitutionRateL) / 12
                    
                for s in range(1,66):
                    self.allfirmSubstitutionsByVendorSector[s-1, aFirm.country]+= (aFirm.capital*self.substitutionRateL*\
                                                                                   self.invSharesNp[aFirm.sector-1,s]) / 12
            
            # general case
            else:                                        
                self.firmSubstitutions[aFirm.country] += (aFirm.capital*self.substitutionRate) / 12
                self.allfirmSubstitutionsByBuyerSector[aFirm.sector-1, aFirm.country] += (aFirm.capital*self.substitutionRate) / 12

                for s in range(1,66):
                    self.allfirmSubstitutionsByVendorSector[s-1, aFirm.country]+= (aFirm.capital*self.substitutionRate*\
                                                                                 self.invSharesNp[aFirm.sector-1,s]) / 12


        ###################################################################################################
        ###################### INTERMEDIATE GOODS PURCHASING NEEDS BY FIRM IN EACH RANK ###################
        ###################################################################################################

        # this step identifies the desired quantities of intermediate inputs in the case that firms can actually produce their entire added value
        # these quantities will be further compared with the availability of the intermediate inputs
        

            for s in range(64):
                aFirm.intermediateWarehouseRefill[s] = aFirm.targetAddedValue * self.intermediateInputs[s][aFirm.sector-1]\
                                                                                         - aFirm.intermediateWarehouse[s]
                # totalizers to measure this quantity in a matrix sector x country
                self.intermediateTotalDemandBySector[s] +=  aFirm.intermediateWarehouseRefill[s] 
                self.intermediateTotalDemandByCountry[aFirm.country] +=  aFirm.intermediateWarehouseRefill[s] 
                self.intermediateTotalDemandBySectorAndCountry[s][aFirm.country] +=  aFirm.intermediateWarehouseRefill[s] 

                
        #print("Demand", self.intermediateTotalDemandBySector)

        # we compared the results of the model computing both the intermediate goods demand and the supply (computed either by sector and by country)
        # follow Intermediate Goods Consistency test at https://prsm.uk/prsm.html?room=UTZ-JPY-SWQ-LJH
        # remember that quantities have been adjusted for the propFactor and from months to year ( x12)
        # the difference between supply and demand of Intermediate goods represents its Export
        

        
        ###################################################################################################
        ############################## COMPUTING GDP BY COUNTRY BY RANK ################################### 
        ###################################################################################################
        # from now we are out of the aFirm "for" cycle because we are observing aggregate consumptions

        self.gdp_eu27 =[0] * 27
        for aCountry in range(27):
            self.gdp_eu27[aCountry] = int(self.markupTentative *\
                                         (self.countEmployeeCompensations[aCountry] * self.propFactor / 1000000\
                                          + self.countCapitalR[aCountry] * self.propFactor / 1000))
            #self.markupTentative * (self.countEmployeeCompensations[aCountry] + self.countCapitalR[aCountry])

        # for jjj in range (27):
            # print(jjj+1,self.eu_countries[jjj],self.gdp_eu27[jjj])


        ###################################################################################################
        ####################### EVALUATING PRIVATE AND GOVERNMENT CONSUMPTIONS ############################ 
        ###################################################################################################
        
        self.private_consumption =[0] * 27
        self.gov_consumption =[0] * 27


        adjusting_consumptions_with_d_imports_private = 1 - 476/7284
        adjusting_consumptions_with_d_imports_gov = 1 - 44/3375
        # these values are copied and rounded from table naio 2022 (data_import.ipynb) to get the parameters providing the adjustment 

        for aCountry in range(27):
            self.private_consumption[aCountry] = self.consumptionShares.iloc[aCountry, -2] * self.gdp_eu27[aCountry]  
            #* adjusting_consumptions_with_d_imports_private ## dubbio: eliminiamo le importazioni dirette per ordinare la prod alle firm?
            self.gov_consumption[aCountry] = self.consumptionShares.iloc[aCountry , - 1] * self.gdp_eu27[aCountry]  
            #* adjusting_consumptions_with_d_imports_gov

        # illuminazione(??): quello che ordiniamo alle imprese produttrici di beni di consumo è la fetta di added vaoue prodotto da loro

        # to calculate internal consumption we introduce two adjusting_consumptions_with_d_imports parameters 
        # neglecting the consumption goods which are directly exported

        # notice that the results produced by the model are very consistent among them
        # let us consider Austria as an example: 
        # simulated gdp = 424188; eurostat gdp (from self.consumptionShares) = 449382.2 --> ratio = 0.94
        # sim private consumption = 16833 * 12 =  201996 ; eurostat private consumption (from self.consumptionShares) = 228963.2 -->  ratio = 0.94
        # meaning that they are perfectly in fit reminding that direct exports are a share included in the general private consumption

        # lastly, notice that direct exports may represent significant tools for policy decisions
                

    ###################################################################################################
    ########################## INTERMEDIATE GOODS RANDOM SUPPLIER SELECTION ###########################
    ###################################################################################################
    def acquireSupplierLock(self, supplier):

        if rankNum == 1:
            return

        lockPointer = ctypes.cast(
            self.supplierLocks.ctypes.data
            + supplier.address * self.supplierLocks.dtype.itemsize,
            ctypes.POINTER(ctypes.c_int32),
        )

        self.atomicLockLibrary.acquire_lock(
            lockPointer
        )


    def releaseSupplierLock(self, supplier):

        if rankNum == 1:
            return

        lockPointer = ctypes.cast(
            self.supplierLocks.ctypes.data
            + supplier.address * self.supplierLocks.dtype.itemsize,
            ctypes.POINTER(ctypes.c_int32),
        )

        self.atomicLockLibrary.release_lock(
            lockPointer
        )


    def purchaseFromSupplier(self, supplier, requested):

        # Fast path for a single rank: no concurrent access is possible.
        if rankNum == 1:

            available = max(
                0.0,
                supplier.outputWarehouse,
            )

            purchased = min(
                requested,
                available,
            )

            remaining = available - purchased

            if purchased > 0.0:
                supplier.outputWarehouse = remaining

            return purchased, available, remaining

        # Multirank path: protect the shared warehouse transaction.
        self.acquireSupplierLock(
            supplier
        )

        try:

            available = max(
                0.0,
                supplier.outputWarehouse,
            )

            purchased = min(
                requested,
                available,
            )

            remaining = available - purchased

            if purchased > 0.0:
                supplier.outputWarehouse = remaining

        finally:

            self.releaseSupplierLock(
                supplier
            )

        return purchased, available, remaining

    def selectSuppliers(self):

        # All ranks complete production and demand before cross-rank purchases start.
        self.sharedComm.Barrier()

        #subset of firms by countries and sectors are conveniently prepared in self.createFirmSubsets
        #adding also the ghosts if created (rankNum > 1)
        
        # Statistics
        countSuccessfulRefillOperations = 0
        countSuccessfulLocalRefillOperations = 0
        countTotalRefillOperations = 0
        countTotalLocalRefillOperations = 0

        if not params["stop_technical_print"]:
            print("\nBefore refill")
            printDemandOfferRefill(outAddedV=True)

        imports = [0.0] * 64

        if params["restocking"] == "n":
            print("\nRestocking disabled")
            return

        local_preference = float(
            params["localPreference"]
        )

        max_attempts = (
            int(params["max_refill_repeat"]) + 1
        )

        import_enabled = bool(
            params["import"]
        )

        randomize_country_order = bool(
            params.get(
                "randomize_country_order",
                False,
            )
        )

        if local_preference <= 0.0:
            raise ValueError(
                "localPreference must be strictly positive."
            )

        # Generate the country order once for the current period.
        # The same order is used for every sector.
        if randomize_country_order:

            country_order = rng_np.permutation(27)

        else:

            country_order = np.arange(27)

        print(
            "\nCountry processing order",
            [
                int(country)
                for country in country_order
            ],
        )

        # Fenwick tree for weighted supplier selection.
        class FenwickTree:

            def __init__(self, values):

                self.n = len(values)
                self.tree = [0.0] * (self.n + 1)

                for index, value in enumerate(
                    values,
                    start=1,
                ):
                    self.tree[index] += value

                    parent = (
                        index + (index & -index)
                    )

                    if parent <= self.n:
                        self.tree[parent] += (
                            self.tree[index]
                        )

            def update(self, index, difference):

                index += 1

                while index <= self.n:
                    self.tree[index] += difference
                    index += index & -index

            def prefix_sum(self, end):

                # Sum over the interval [0, end).
                total = 0.0
                index = end

                while index > 0:
                    total += self.tree[index]
                    index -= index & -index

                return total

            def range_sum(self, start, end):

                return (
                    self.prefix_sum(end)
                    - self.prefix_sum(start)
                )

            def total(self):

                return self.prefix_sum(self.n)

            def find_by_mass(self, mass):

                # Find the first position whose cumulative
                # weight is greater than mass.
                index = 0

                bit = 1 << (
                    self.n.bit_length() - 1
                )

                while bit:

                    next_index = index + bit

                    if (
                        next_index <= self.n
                        and self.tree[next_index] <= mass
                    ):
                        mass -= self.tree[next_index]
                        index = next_index

                    bit >>= 1

                if index >= self.n:
                    index = self.n - 1

                return index

        # Process one intermediate-input sector at a time.
        for aSector, sector_suppliers in enumerate(
            self.firmListBySectorWithProductionTypeInt
        ):

            if not sector_suppliers:
                continue

            # Group suppliers by country.
            suppliers_by_country = [
                [] for _ in range(27)
            ]

            for supplier in sector_suppliers:

                suppliers_by_country[
                    supplier.country
                ].append(supplier)

            # Build one supplier list ordered by country.
            suppliers = []

            country_start = [0] * 27
            country_end = [0] * 27

            for country in range(27):

                country_start[country] = len(
                    suppliers
                )

                suppliers.extend(
                    suppliers_by_country[country]
                )

                country_end[country] = len(
                    suppliers
                )

            # Additional-supplier selection weights are based
            # on currently available outputWarehouse.
            available_values = [
                max(
                    0.0,
                    float(
                        supplier.outputWarehouse
                    ),
                )
                for supplier in suppliers
            ]

            warehouse_tree = FenwickTree(
                available_values
            )

            # Connect each supplier object to its position
            # in the Fenwick tree.
            supplier_positions = {
                id(supplier): position
                for position, supplier
                in enumerate(suppliers)
            }

            # Process countries in fixed or randomized order.
            for country_position in country_order:

                aCountry = int(
                    country_position
                )

                buyers = (
                    self.firmListByCountry[
                        aCountry
                    ]
                )

                if not buyers:
                    continue

                local_start = (
                    country_start[aCountry]
                )

                local_end = (
                    country_end[aCountry]
                )

                if len(sector_suppliers) == 0:
                    print("NO SUPPLIERS", "rank", rank, "sector", aSector, "country", aCountry)
                    continue

                if sum(aSupplier.targetAddedValue for aSupplier in sector_suppliers) <= 0:
                    print("ZERO SUPPLIER WEIGHTS", "rank", rank, "sector", aSector,
                    "country", aCountry, "n. suppliers", len(sector_suppliers))
                    continue

                # Select the first supplier through
                # the two-tape matching method.
                (
                    permuted_suppliers,
                    matched_positions,
                    _,
                ) = match_buyers_to_suppliers(
                    buyers,
                    sector_suppliers,
                    aCountry,
                    local_preference,
                    rng_np,
                    "systematic",
                )

                for aBuyer, initial_position in zip(
                    buyers,
                    matched_positions,
                ):

                    refill = (
                        aBuyer
                        .intermediateWarehouseRefill[
                            aSector
                        ]
                    )

                    # Only positive demand can generate
                    # a supplier contact.
                    if refill <= 0.0:
                        continue

                    attempts = 0

                    # ========================================
                    # First supplier: two-tape matching
                    # ========================================

                    supplier = (
                        permuted_suppliers[
                            int(initial_position)
                        ]
                    )

                    is_local = (
                        supplier.country
                        == aCountry
                    )

                    # Count the contact even if the supplier
                    # cannot make a positive delivery.
                    attempts += 1
                    countTotalRefillOperations += 1

                    if is_local:
                        countTotalLocalRefillOperations += 1

                    purchased, available, remaining = self.purchaseFromSupplier(
                        supplier, refill
                    )

                    tree_position = supplier_positions[id(supplier)]

                    if rankNum == 1:

                        # No concurrent modification is possible.
                        warehouse_tree.update(
                            tree_position,
                            -purchased,
                        )

                    else:

                        # Another rank may have changed this warehouse after the
                        # local Fenwick tree was built. Reconcile the local slot.
                        warehouse_tree.update(
                            tree_position,
                            remaining - available_values[tree_position],
                        )

                        available_values[
                            tree_position
                        ] = remaining
                        
                    if purchased > 0.0:
                        countSuccessfulRefillOperations += 1

                        if is_local:
                            countSuccessfulLocalRefillOperations += 1

                        aBuyer.intermediateWarehouse[aSector] += purchased
                        refill -= purchased

                    # ========================================
                    # Additional suppliers:
                    # weighted by available outputWarehouse
                    # ========================================

                    while (
                        refill > 0.0
                        and attempts < max_attempts
                    ):

                        total_supply = (
                            warehouse_tree.total()
                        )

                        if total_supply <= 0.0:
                            break

                        local_supply = (
                            warehouse_tree.range_sum(
                                local_start,
                                local_end,
                            )
                        )

                        foreign_supply = (
                            total_supply
                            - local_supply
                        )

                        # Protection against small
                        # floating-point errors.
                        if local_supply < 0.0:
                            local_supply = 0.0

                        if foreign_supply < 0.0:
                            foreign_supply = 0.0

                        weighted_local_supply = (
                            local_preference
                            * local_supply
                        )

                        weighted_total_supply = (
                            weighted_local_supply
                            + foreign_supply
                        )

                        if weighted_total_supply <= 0.0:
                            break

                        # Choose between domestic suppliers
                        # and suppliers in other EU27 countries.
                        choose_local = (
                            rng_np.random()
                            * weighted_total_supply
                            < weighted_local_supply
                        )

                        if (
                            choose_local
                            and local_supply > 0.0
                        ):

                            mass_before_local = (
                                warehouse_tree.prefix_sum(
                                    local_start
                                )
                            )

                            local_mass = (
                                rng_np.random()
                                * local_supply
                            )

                            selected_position = (
                                warehouse_tree
                                .find_by_mass(
                                    mass_before_local
                                    + local_mass
                                )
                            )

                        elif foreign_supply > 0.0:

                            # Foreign EU27 suppliers occupy
                            # the intervals before and after
                            # the buyer's domestic interval.
                            supply_before_local = (
                                warehouse_tree.prefix_sum(
                                    local_start
                                )
                            )

                            foreign_mass = (
                                rng_np.random()
                                * foreign_supply
                            )

                            if (
                                foreign_mass
                                < supply_before_local
                            ):

                                selected_mass = (
                                    foreign_mass
                                )

                            else:

                                selected_mass = (
                                    warehouse_tree
                                    .prefix_sum(
                                        local_end
                                    )
                                    + foreign_mass
                                    - supply_before_local
                                )

                            selected_position = (
                                warehouse_tree
                                .find_by_mass(
                                    selected_mass
                                )
                            )

                        elif local_supply > 0.0:

                            # Only domestic supply remains.
                            mass_before_local = (
                                warehouse_tree.prefix_sum(
                                    local_start
                                )
                            )

                            local_mass = (
                                rng_np.random()
                                * local_supply
                            )

                            selected_position = (
                                warehouse_tree
                                .find_by_mass(
                                    mass_before_local
                                    + local_mass
                                )
                            )

                        else:
                            break

                        supplier = suppliers[
                            selected_position
                        ]

                        is_local = (
                            supplier.country
                            == aCountry
                        )

                        # Count every additional supplier
                        # contact, successful or unsuccessful.
                        attempts += 1
                        countTotalRefillOperations += 1

                        if is_local:
                            countTotalLocalRefillOperations += 1

                        purchased, available, remaining = self.purchaseFromSupplier(
                            supplier, refill
                        )

                        if rankNum == 1:

                            # No concurrent modification is possible.
                            warehouse_tree.update(
                                selected_position,
                                -purchased,
                            )

                        else:

                            # Reconcile the local Fenwick slot with the actual
                            # shared warehouse value observed under the CAS lock.
                            warehouse_tree.update(
                                selected_position,
                                remaining - available_values[selected_position],
                            )

                            available_values[
                                selected_position
                            ] = remaining
                            
                        if purchased <= 0.0:
                            continue

                        countSuccessfulRefillOperations += 1

                        if is_local:
                            countSuccessfulLocalRefillOperations += 1

                        aBuyer.intermediateWarehouse[aSector] += purchased
                        refill -= purchased

                    # Cover residual demand through
                    # extra-EU imports.
                    if (
                        refill > 0.0
                        and import_enabled
                    ):

                        aBuyer.intermediateWarehouse[
                            aSector
                        ] += refill

                        imports[aSector] += refill
                        refill = 0.0

                    aBuyer.intermediateWarehouseRefill[
                        aSector
                    ] = refill

        failedRefillOperations = (
            countTotalRefillOperations
            - countSuccessfulRefillOperations
        )

        failedLocalRefillOperations = (
            countTotalLocalRefillOperations
            - countSuccessfulLocalRefillOperations
        )

        if not params["stop_technical_print"]: print(
            "\n\ncountTotalRefillOperations",
            countTotalRefillOperations,
            "countTotalLocalRefillOperations",
            countTotalLocalRefillOperations,
            "\ncountSuccessfulRefillOperations",
            countSuccessfulRefillOperations,
            "countSuccessfulLocalRefillOperations",
            countSuccessfulLocalRefillOperations,
            "\ncountFailedRefillOperations",
            failedRefillOperations,
            "countFailedLocalRefillOperations",
            failedLocalRefillOperations,
            flush=True,
        )

        # Print residual demand and supply before moving
        # output to the potential-export warehouses.
        if not params["stop_technical_print"]:
            print("\nAfter refill")
            printDemandOfferRefill(outAddedV=False)

            print(
            "\nImports",
            [
                (
                    sector + 1,
                    round(
                        value / 1_000_000,
                        2,
                    ),
                )
                for sector, value
                in enumerate(imports)
            ],
        )

            print(
            "\nTotal imports",
            round(
                sum(imports) / 1_000_000,
                2,
            ),
        )

        # All cross-rank purchases must finish before owners move residual
        # output to the potential extra-EU export warehouse.
        self.sharedComm.Barrier()

        potentialExtraEU27Export = 0.0

        # Only locally owned Firms perform this operation.
        for supplier in context.agents(agent_type=0):

            if supplier.productionType != "Int":
                continue

            residual_output = max(0.0, supplier.outputWarehouse)

            if residual_output <= 0.0:
                supplier.outputWarehouse = 0.0
                continue

            supplier.warehouseForPotentialExtraEU27export += residual_output
            potentialExtraEU27Export += residual_output
            supplier.outputWarehouse = 0.0

        self.sharedComm.Barrier()

        # Current-period potential extra-EU export flow.
        if not params["stop_technical_print"]: print(
            "\nPotential ExtraEU27 export",
            round(
                potentialExtraEU27Export
                / 1_000_000,
                2,
            ),
        ) 

        
    ###################################################################################################
    ################################# AGGREGATING RESULTS BY COUNTRY ##################################
    ###################################################################################################
    def aggregatingResultsByCountry(self):
        for j in range(27):
            self.countWorkers[27] += self.countWorkers[j]
            self.countEmployeeCompensations[27] += self.countEmployeeCompensations[j] 
            self.countCapitalR[27] += self.countCapitalR[j] 
            self.countActualAddedValue[27] += self.countActualAddedValue[j]
            self.countAFF_Firms[27] += self.countAFF_Firms[j]
            self.countAFF_WorkerNumber[27] += self.countAFF_WorkerNumber[j] 
            self.intermediateTotalSupplyByCountry[27] += self.intermediateTotalSupplyByCountry[j]
            self.intermediateTotalDemandByCountry[27] += self.intermediateTotalDemandByCountry[j]
            self.firmSubstitutions[27] += self.firmSubstitutions[j] 

            self.allfirmSubstitutionsByVendorSector[:, 27] += self.allfirmSubstitutionsByVendorSector[:,j]
            self.allfirmSubstitutionsByBuyerSector[:, 27] += self.allfirmSubstitutionsByBuyerSector[:,j]

            # remember that you have to add the aggregation by sector!!!!!!!!!!!!!!!!!!!!!!


    # aggregate results over ranks
    def aggregatingRanks(self):
        if rankNum == 1: return

        # The number of firms is constant and has to be reduced only once.
        if self.count_months == 0:
            totalFirmNumber = comm.reduce(self.countActualExPostFirmNumber, op=MPI.SUM, root=0)
            if rank == 0:
                self.countActualExPostFirmNumber = totalFirmNumber

        # Pack all monthly results in one contiguous NumPy array.
        # In this way only one MPI.Reduce is required each month.
        if self.detailedTiming:
            packWallStart = time.perf_counter()
            packCpuStart = time.process_time()
        localResults = np.concatenate((
            np.asarray(self.countWorkers, dtype=np.float64),
            np.asarray(self.countEmployeeCompensations, dtype=np.float64),
            np.asarray(self.countCapitalR, dtype=np.float64),
            np.asarray(self.countAFF_Firms, dtype=np.float64),
            np.asarray(self.countAFF_WorkerNumber, dtype=np.float64),
            np.asarray(self.intermediateTotalSupplyBySector, dtype=np.float64),
            np.asarray(self.intermediateTotalSupplyByCountry, dtype=np.float64),
            np.asarray(self.intermediateTotalDemandBySector, dtype=np.float64),
            np.asarray(self.intermediateTotalDemandByCountry, dtype=np.float64),
            np.asarray(self.firmSubstitutions, dtype=np.float64),
            np.asarray(self.allfirmSubstitutionsByVendorSector, dtype=np.float64).ravel(),
            np.asarray(self.allfirmSubstitutionsByBuyerSector, dtype=np.float64).ravel(),
            np.asarray(self.intermediateTotalSupplyBySectorAndCountry, dtype=np.float64).ravel(),
            np.asarray(self.intermediateTotalDemandBySectorAndCountry, dtype=np.float64).ravel(),
            np.asarray(self.countActualAddedValue, dtype=np.float64)
        ))
        if self.detailedTiming:
            self.reduceTimingWall["pack"] = time.perf_counter() - packWallStart
            self.reduceTimingCpu["pack"] = time.process_time() - packCpuStart

        if rank == 0:
            totalResults = np.empty_like(localResults)
        else:
            totalResults = None

        if self.detailedTiming:
            reduceWallStart = time.perf_counter()
            reduceCpuStart = time.process_time()
        comm.Reduce(localResults, totalResults, op=MPI.SUM, root=0)
        if self.detailedTiming:
            self.reduceTimingWall["reduce"] = time.perf_counter() - reduceWallStart
            self.reduceTimingCpu["reduce"] = time.process_time() - reduceCpuStart

        if rank != 0: return

        # Unpack the reduced results on rank 0.
        if self.detailedTiming:
            unpackWallStart = time.perf_counter()
            unpackCpuStart = time.process_time()
        p = 0

        self.countWorkers = totalResults[p:p+28].astype(np.int64).tolist()
        p += 28
        self.countEmployeeCompensations = totalResults[p:p+28].tolist()
        p += 28
        self.countCapitalR = totalResults[p:p+28].tolist()
        p += 28
        self.countAFF_Firms = totalResults[p:p+28].astype(np.int64).tolist()
        p += 28
        self.countAFF_WorkerNumber = totalResults[p:p+28].astype(np.int64).tolist()
        p += 28

        self.intermediateTotalSupplyBySector = totalResults[p:p+64].tolist()
        p += 64
        self.intermediateTotalSupplyByCountry = totalResults[p:p+28].tolist()
        p += 28
        self.intermediateTotalDemandBySector = totalResults[p:p+64].tolist()
        p += 64
        self.intermediateTotalDemandByCountry = totalResults[p:p+28].tolist()
        p += 28
        self.firmSubstitutions = totalResults[p:p+28].tolist()
        p += 28

        self.allfirmSubstitutionsByVendorSector = totalResults[p:p+65*28].reshape(65,28)
        p += 65*28
        self.allfirmSubstitutionsByBuyerSector = totalResults[p:p+65*28].reshape(65,28)
        p += 65*28

        self.intermediateTotalSupplyBySectorAndCountry = totalResults[p:p+64*28].reshape(64,28).tolist()
        p += 64*28
        self.intermediateTotalDemandBySectorAndCountry = totalResults[p:p+64*28].reshape(64,28).tolist()
        p += 64*28

        self.countActualAddedValue = totalResults[p:p+28].tolist()
        if self.detailedTiming:
            self.reduceTimingWall["unpack"] = time.perf_counter() - unpackWallStart
            self.reduceTimingCpu["unpack"] = time.process_time() - unpackCpuStart

    # show results
    def showResults(self):
        
        if rank != 0: return

        self.propFactor = self.Eu_FirmNumber / self.countActualExPostFirmNumber # redefined here to be updated with rankNum > 1

        print("\n", "MONTH",  (self.count_months % 12) + 1 , "YEAR", (self.count_months // 12) + 1)
        print("\nLocal preference", params['localPreference'], "max_refill_repeat", params['max_refill_repeat'])

        
        generateOutput("totEU27", sys.stdout, self.AFF_FirmNumber, self.countAFF_Firms[27], self.propFactor, self.AFF_WorkerNumber,\
                    self.countAFF_WorkerNumber[27], self.Eu_FirmNumber, self.countActualExPostFirmNumber, self.Eu_WorkerNumber,\
                    self.countWorkers[27], self.Eu_EmployeeCompensations, self.countEmployeeCompensations[27], self.Eu_GDP_withVAT2022,\
                    self.Eu_AddedValue2022, self.countCapitalR[27], self.countActualAddedValue[27], self.markupTentative, self.Eu_Intermediate2022,\
                       self.intermediateTotalDemandByCountry[27], self.substitutionRate, self.substitutionRateL, self.firmSubstitutions[27],\
                       self.allfirmSubstitutionsByBuyerSector[:,27], self.simulationFirmsExAnteNumber, self.allfirmSubstitutionsByVendorSector[:,27])

        if self.selected_countries == []: return

        for aCountry in self.selected_countries:
            if os.path.exists(self.eu_countries[aCountry]+".txt") and self.count_months == 1: os.remove(self.eu_countries[aCountry]+".txt")
            
            with open(self.eu_countries[aCountry]+'.txt', 'a', newline='') as txtfile:              
                #print(txtfile)
                print("MONTH", (self.count_months % 12) + 1, file = txtfile)
                generateOutput(self.eu_countries[aCountry], txtfile, self.AFF_FirmNumber,
                                self.countAFF_Firms[aCountry], self.propFactor,
                                self.AFF_WorkerNumber, self.countAFF_WorkerNumber[aCountry],
                                self.Eu_FirmNumber, self.countActualExPostFirmNumber,
                                self.Eu_WorkerNumber, self.countWorkers[aCountry],
                                self.Eu_EmployeeCompensations, self.countEmployeeCompensations[aCountry],
                                self.Eu_GDP_withVAT2022, self.Eu_AddedValue2022,
                                self.countCapitalR[aCountry], 
                                self.countActualAddedValue[aCountry],
                                self.markupTentative,
                                self.Eu_Intermediate2022, self.intermediateTotalDemandByCountry[aCountry],
                                self.substitutionRate, self.substitutionRateL,
                                self.firmSubstitutions[aCountry],
                                self.allfirmSubstitutionsByBuyerSector[:, aCountry],
                                self.simulationFirmsExAnteNumber,
                                self.allfirmSubstitutionsByVendorSector[:, aCountry]
                              )

        # ptpt if rank == 0: 
        print("\n", 'Add. Val. basic p. by country', "\n")
        for aCountry in range(len(self.eu_countries)):
            generateGDPOutput(self.eu_countries[aCountry], sys.stdout, self.propFactor, self.countActualAddedValue[aCountry])
                        #self.countEmployeeCompensations[aCountry], self.countCapitalR[aCountry], self.markupTentative)

    
    #finish
    def finish(self):
        print("\n\n")
        print(f"Rank {rank}, global time {T()}, cpu time {Tc()}")
        print("\nRank ", rank, "concluded")
    
    def start(self):
        runner.execute()

def run():

    model = Model() 
    model.start()
    model.sharedComm.Barrier()
    model.supplierLockWin.Free()
    model.targetAddedValueWin.Free()
    model.outputWarehouseWin.Free()
    model.sharedComm.Free()
    return model # to access model when the execution is finished
    
model=run() #run() returns model, useful to access all the variables of the model

#file name: ff_with_class_limits.csv

rank 0 tableOfInvestmentShare 2
rank 0 how many firms 1000000
rank 0 use smart capital 1
1000031 firms created in rank  0
propFactor 42.25056423250879 in rank 0

Country processing order [11, 3, 17, 2, 19, 6, 26, 24, 5, 7, 4, 23, 0, 1, 22, 10, 18, 25, 16, 9, 14, 20, 15, 12, 21, 8, 13]
TIMING rank   0 month   1 | wall: supply 11.822180  demand 36.612302  country 0.000089  suppliers 165.903845  ranks 0.000011  pack 0.000000  reduce 0.000000  unpack 0.000000 | cpu: supply 11.791163  demand 36.569087  country 0.000089  suppliers 164.709056  ranks 0.000010  pack 0.000000  reduce 0.000000  unpack 0.000000

 MONTH 1 YEAR 1

Local preference 20 max_refill_repeat 2

 totEU27 

Markup tentative 25.0%
                                    EU 27           simulation        simulation to
                                                                           EU scale
     AFF Firm Number             10000000               236736             10002229
   AFF Worker Number              9000000       

---------  
   
Italy simulated GDP per month: 133108  
Italy GDP in 2022: 1946479 --> 1946479 / (12 * 1.22) = 132956.2 ca. to get monthly at basic price GDP 

**to run the code in a multi-rank way, from the terminal launch:**  

mpirun -n x ipython modelEU27.ipynb

**using Ryzen Threadripper:**

mpirun -n X -bind-to core ipython modelEU27.ipynb

*best choice: X = 40*

**to access the contents of the program after the execution**

%whos - gives the list of the variables  
context - is the address of the context; e.g., context.agent((0,0,0)).outputWarehouse  
model - is the address of the model, returned by run(); e.g., model.eu_countries